# Getting Started with Claude Haiku 5.5 on Amazon Bedrock

**According to Anthropic, Claude Haiku 5.5 is the fastest and most efficient model in the Claude 5.5 family — built for subagents and high-volume, latency-sensitive work.**

Claude Haiku 5.5 is the most capable Haiku model across coding, tool use, computer use, and agentic tasks. It is also the **first Haiku model with effort controls**, so you can tune cost against intelligence for each task instead of picking one setting for an entire workload.

It is available on Amazon Bedrock through **three APIs across two endpoints**, so you can integrate with the Anthropic SDK, boto3, or the AWS CLI.

## What's new in Claude Haiku 5.5
- **Effort controls** — a first for the Haiku family; tune cost against intelligence per task.
- **Agentic coding and multi-step tool use** — works as a subagent, not just a single-shot model.
- **High-resolution image support** — a strong computer use subagent for repetitive browser and desktop tasks, at a cost that holds up at scale.


## Key capabilities

| Capability | Details |
|---|---|
| **Context window** | 1M  |
| **Max output** | 128k |
| **Knowledge cutoff** | June 2026 |
| **Adaptive thinking** | Supported, with effort levels — a first for the Haiku family, can be disabled, effort capped at high when thinking is disabled, default effort level is medium |
| **Vision** | High-resolution image input |
| **Computer use** | Supported |

## Model ID and API options

| API | Endpoint | SDK | Model ID | When to use |
|---|---|---|---|---|
| **Messages** | `bedrock-runtime` | `anthropic` | `global.anthropic.claude-haiku-5-5` | Recommended. Native Anthropic format, widest Region coverage. |
| **Messages** | `bedrock-mantle` | `anthropic[bedrock]` | `anthropic.claude-haiku-5-5` | us-east-1 and ap-southeast-4 only. |
| **InvokeModel** | `bedrock-runtime` | boto3 | `global.anthropic.claude-haiku-5-5` | Bedrock-native. The only API that reports thinking tokens. |
| **Converse** | `bedrock-runtime` | boto3 | `global.anthropic.claude-haiku-5-5` | Unified multi-model API; simplest integration. |

`bedrock-runtime` takes the **geo-prefixed CRIS** model ID. `bedrock-mantle` takes the **bare** model ID. 

**Availability:** US (`us.`), EU (`eu.`), Australia (`au.`), Japan (`jp.`), and Global (`global.`) CRIS inference profiles on `bedrock-runtime`.  Also available through Claude Platform on AWS in North America. See the [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-anthropic.html) for the full Region list.


---

## 1. Setup and configuration

**Prerequisites:**
1. An active AWS account with Amazon Bedrock access to Claude Haiku 5.5.
2. AWS CLI installed and configured.
3. Python 3.10+.
4. `pip install boto3 "anthropic[bedrock]" "anthropic" aws_bedrock_token_generator`.
5. IAM permissions `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream`.

## Regions

On `bedrock-runtime` the model is served through cross-Region inference, so use a geo-prefixed
CRIS model ID rather than the bare one. The launch blog lists US (`us.`), EU (`eu.`),
Australia (`au.`), Japan (`jp.`), and Global (`global.`).

It is also available through Claude Platform on AWS in North America. Confirm the current list in
the [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-anthropic.html).

In [ ]:
%pip install --quiet --upgrade boto3 "anthropic[bedrock]" aws_bedrock_token_generator anthropic

In [ ]:
import boto3, json, re, base64, time, math, statistics
from botocore.config import Config

REGION = "eu-central-1"
CRIS_MODEL_ID = "eu.anthropic.claude-haiku-5-5"    # InvokeModel / Converse / Messages on bedrock-runtime
MANTLE_MODEL_ID = "anthropic.claude-haiku-5-5"         # Messages API on bedrock-mantle
MANTLE_REGION = "us-east-1"                            
BASELINE_MODEL_ID = "global.anthropic.claude-haiku-4-5"  # for the cost-per-task comparison in section 6
ANTHROPIC_VERSION = "bedrock-2023-05-31"

# Comparison models for the latency and orchestration sections.
OPUS_MODEL_ID = "global.anthropic.claude-opus-5-5"
SONNET_MODEL_ID = "global.anthropic.claude-sonnet-5-5"

# Haiku 5.5 reasons before it answers, so keep a generous read timeout.
CFG = Config(read_timeout=1200, retries={"max_attempts": 2})

rt = boto3.client("bedrock-runtime", region_name=REGION, config=CFG)
print("boto3", boto3.__version__, "| region", REGION, "| model", CRIS_MODEL_ID)

### Messages API clients

Existing Anthropic SDK code runs against Bedrock by pointing `base_url` at the endpoint's
`/anthropic` path and authenticating with a short-lived Bedrock token. Two shapes, and they
differ in more than the URL — **class, auth, and model-ID form all change**:

| | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| Class | `Anthropic` | `AnthropicBedrockMantle` |
| Auth | Bedrock token via `provide_token` | SigV4 from your AWS credentials |
| Model ID | **CRIS** (`global.anthropic.…`) | **bare** (`anthropic.…`) |
| APIs served | Messages, Converse, InvokeModel | Messages only |

Getting the model-ID form wrong is the most common first error on Bedrock.

In [ ]:
from anthropic import Anthropic
from aws_bedrock_token_generator import provide_token

# On bedrock-runtime (CRIS model ID)
 
rt_messages = Anthropic(base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/anthropic",
                        api_key=provide_token(region=REGION))
print("Messages (runtime) endpoint:", rt_messages.base_url)

# On bedrock-mantle (bare model ID, in-Region endpoint at bedrock-mantle.{region}.api.aws)
mantle = None
try:
    from anthropic import AnthropicBedrockMantle
    mantle = AnthropicBedrockMantle(aws_region=MANTLE_REGION)
    print(f"Messages (mantle, {MANTLE_REGION}) endpoint:", mantle.base_url)
except Exception as e:
    # The Mantle catalog can lag the runtime catalog for a new model; the runtime paths still work.
    print("Mantle client unavailable on this account/Region:", type(e).__name__, str(e)[:140])

---

## 2. Basic usage

Three API paths reach Haiku 5.5. `bedrock-runtime` serves all three — Messages, Converse, and
InvokeModel; `bedrock-mantle` serves the Messages API.

**Select content blocks by type, never by index.** Haiku 5.5 may return a reasoning block before
the `text` block, so `content[0]` is not reliably the answer. Remember too that `max_tokens`
bounds thinking *and* response text together — leave headroom for both.

In [ ]:
question = (
    "What are three key considerations when designing "
    "a distributed system for high availability?"
)

print("Messages API:\n")
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID,
        max_tokens=2048,
        messages=[{"role": "user", "content": question}],
    )
    # Select by type: a thinking block may come first.
    text = next(b.text for b in message.content if b.type == "text")
    print(text)
    print(f"\nBlock types returned: {[b.type for b in message.content]}")
    print(f"Usage: input={message.usage.input_tokens}, output={message.usage.output_tokens}")
except Exception as e:
    print(f"Error: {e}")

The same request on **InvokeModel** and **Converse**. 

In [ ]:
# InvokeModel (native Anthropic JSON body)
print("InvokeModel:\n")
try:
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": question}],
    }
    r = rt.invoke_model(
        modelId=CRIS_MODEL_ID, contentType="application/json",
        accept="application/json", body=json.dumps(body),
    )
    res = json.loads(r["body"].read())
    print(next(b["text"] for b in res["content"] if b["type"] == "text")[:300] + " ...")
    print(f"\nBlock types returned: {[b['type'] for b in res['content']]}")
except Exception as e:
    print(f"Error: {e}")

# Converse (unified API)
print("\n\nConverse:\n")
try:
    r = rt.converse(
        modelId=CRIS_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": question}]}],
        inferenceConfig={"maxTokens": 2048},
    )
    blocks = r["output"]["message"]["content"]
    # A reasoningContent block has no top-level "text" key, so this filter is safe.
    print("\n".join(b.get("text", "") for b in blocks if "text" in b)[:300] + " ...")
    print(f"\nBlock keys returned: {[list(b)[0] for b in blocks]}")
    print(f"Usage: {r['usage']}")
except Exception as e:
    print(f"Error: {e}")

---

## 3. Adaptive thinking and effort

Haiku 5.5 is the **first Haiku model with effort controls**. Adaptive thinking lets Claude
reason more deeply when a task needs it and respond quickly when it does not, and effort gives
you a dial over how much of that reasoning to buy per request. 

**How it works:**
- `thinking.type` is `"adaptive"` by default, so Claude decides how much to think based on task
  complexity. **Thinking can be disabled**. Effort capped at high when thinking is disabled
- Control depth with `output_config.effort`. On the Messages API, `output_config` is passed
  through `extra_body`; on InvokeModel it goes at the top level of the body; on Converse it goes
  in `additionalModelRequestFields`.
- Thinking text is **omitted by default** — you get a block with empty text and a long
  signature. Set `thinking.display` to `"summarized"` to read it.

### Effort levels

Default effort level is medium when effort level is not provided.

| Effort | Behavior | Best for |
|--------|----------|----------|
| **max** | Exhaustive analysis, cost secondary | Hardest problems |
| **xhigh** | Maximum reasoning depth | Most complex analytical work |
| **high** | Deep reasoning | Complex tasks |
| **medium** | Balanced | Mixed workloads |
| **low** | Minimal thinking, prioritizes speed | Simple, high-volume, cost-sensitive work |




In [ ]:
# Messages API: output_config goes in extra_body.
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID,
        max_tokens=8000,
        thinking={"type": "adaptive", "display": "summarized"},
        extra_body={"output_config": {"effort": "low"}},
        messages=[{"role": "user", "content":
                   "A queue has 3 consumers processing 120 messages/minute each. "
                   "Arrival rate is 500 messages/minute. What happens, and what is "
                   "the minimum number of consumers needed? Be brief."}],
    )
    for block in message.content:
        if block.type == "thinking":
            if block.thinking:
                print(f"[Thinking summary]\n{block.thinking}\n")
            else:
                print(f"[Thinking omitted - signature {len(block.signature)} chars]\n")
        elif block.type == "text":
            print(block.text)
    details = getattr(message.usage, "output_tokens_details", None)
    thinking_tokens = getattr(details, "thinking_tokens", 0) if details else 0
    print(f"\nUsage: input={message.usage.input_tokens}, "
          f"output={message.usage.output_tokens}, thinking={thinking_tokens}")
except Exception as e:
    print(f"Error: {e}")

### What does Haiku 5.5 actually accept?

Sampling parameters not supported: temperature, top_p, or top_k set to any non-default value returns a 400 error. Remove them from all calls.

Thinking type enabled with budget tokens is also not supported.



In [ ]:

probes = {
    "thinking disabled":        dict(thinking={"type": "disabled"}),
    "thinking enabled+budget":  dict(thinking={"type": "enabled", "budget_tokens": 4096}),
    "temperature":              dict(temperature=0.5),
    "top_p":                    dict(top_p=0.9),
    "top_k":                    dict(top_k=40),
    "effort=low":               dict(extra_body={"output_config": {"effort": "low"}}),
    "effort=medium":            dict(extra_body={"output_config": {"effort": "medium"}}),
    "effort=high":              dict(extra_body={"output_config": {"effort": "high"}}),
    "effort=xhigh":             dict(extra_body={"output_config": {"effort": "xhigh"}}),
    "effort=max":               dict(extra_body={"output_config": {"effort": "max"}}),
}

print(f"{'parameter':<26}result")
print("-" * 78)
for label, kwargs in probes.items():
    try:
        rt_messages.messages.create(
            model=CRIS_MODEL_ID, max_tokens=32,
            messages=[{"role": "user", "content": "Say OK."}], **kwargs,
        )
        print(f"{label:<26}accepted")
    except Exception as e:
        print(f"{label:<26}rejected - {type(e).__name__}: {str(e)[:90]}")


### Comparing effort levels

Effort trades tokens for depth. The cell below runs the same prompt across every effort level
that the probe accepted and reports tokens and latency, so you can pick a tier on evidence.

Measurement is on **InvokeModel** deliberately: Converse does not report thinking tokens, and
without them the token counts are not interpretable.

In [ ]:
def effort_sample(effort_level, prompt, max_tokens=8000):
    """One InvokeModel call at a given effort. Returns tokens and wall-clock latency."""
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": max_tokens,
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": effort_level},
        "messages": [{"role": "user", "content": prompt}],
    }
    t0 = time.perf_counter()
    r = rt.invoke_model(modelId=CRIS_MODEL_ID, body=json.dumps(body))
    elapsed = time.perf_counter() - t0
    res = json.loads(r["body"].read())
    usage = res.get("usage", {})
    text = next((b["text"] for b in res["content"] if b["type"] == "text"), "")
    return {
        "input": usage.get("input_tokens", 0),
        "output": usage.get("output_tokens", 0),
        "thinking": usage.get("output_tokens_details", {}).get("thinking_tokens", 0),
        "seconds": elapsed,
        "chars": len(text),
    }

prompt = (
    "A team runs 40 Lambda functions behind one API Gateway. p99 latency jumped from "
    "120ms to 900ms after a deploy, but only for 5% of requests. List the three most "
    "likely causes in priority order, with the single check you would run for each. "
    "Under 120 words."
)

print(f"{'effort':<16}{'input':>7}{'thinking':>10}{'output':>8}{'chars':>8}{'seconds':>9}")
print("-" * 58)
for effort in ["low", "medium", "high", "xhigh", "max"]:
    try:
        s = effort_sample(effort, prompt)
        print(f"{effort:<16}{s['input']:>7}{s['thinking']:>10}{s['output']:>8}"
              f"{s['chars']:>8}{s['seconds']:>8.2f}s")
    except Exception as e:
        print(f"{effort:<16} skipped - {type(e).__name__}: {str(e)[:50]}")

print("\nFor Haiku's target workloads the lower tiers are usually the right default.")
print("Move up only when a task genuinely needs the depth.")

---

## 4. Streaming

Streaming delivers tokens as they are generated. For interactive applications it is the
difference between a response that feels instant and one that feels slow, even at identical
total latency — which is why it matters more for Haiku than for the larger models.

In [ ]:
print("Messages API streaming:\n")
try:
    with rt_messages.messages.stream(
        model=CRIS_MODEL_ID,
        max_tokens=512,
        messages=[{"role": "user", "content": "List five best practices for API security."}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

---

## 5. Latency compared to Opus 5.5 and Sonnet 5.5

Haiku 5.5 is built for latency-sensitive, high-volume work. This section measures that directly
rather than asking you to take it on faith — but a latency benchmark is easy to run in a way
that measures the wrong thing, so note the controls:

- **Effort is pinned, not left to defaults.** Each model has its own default effort, so an
  unconfigured comparison compares defaults rather than models. We set `low` across all three:
  the tier Haiku's target workloads actually run at.
- **The prompt forces a one-word answer.** A verbose model otherwise looks slow when it is only
  being wordy. Constraining output length is what makes the latencies comparable.
- **Thinking tokens are reported alongside.** Adaptive thinking is on by default, and a model
  that thinks 400 tokens will lose to one that thinks 40 regardless of raw speed. If the
  thinking column diverges sharply, you are reading a reasoning difference, not a speed one.
- **Requests are interleaved round-robin.** Running 50 of one model and then 50 of the next lets
  a transient regional slowdown land entirely on whichever model ran first.
- **The first call per model is discarded.** It carries TLS and client setup.
- **Timing runs on a no-retry client.** The shared `rt` client allows 2 attempts, which is
  correct for production but wrong for measurement: a silent retry gets recorded as one very
  slow call and misread as model latency.

> **Cost:** at `N_REQUESTS = 50` this cell makes **150 billable calls** plus 3 warm-ups. Lower
> `N_REQUESTS` to rehearse it.

> **Note:** These are *your* numbers — one Region, one account, one prompt, at one moment. They
> are not a benchmark claim and they will move with Region, quota, and time of day. At n=50, p95
> is the 3rd-slowest sample, so treat it as directional. Overlapping min–max ranges mean the
> difference is noise, not a result.

In [ ]:
from collections import defaultdict

# Measurement client: same settings as CFG but with retries off. The shared `rt` client allows
# 2 attempts, which is right for production but wrong for timing - a silent retry would be
# recorded as one very slow call and read as model latency.
MEASURE_CFG = Config(read_timeout=1200, retries={"max_attempts": 1})
rt_measure = boto3.client("bedrock-runtime", region_name=REGION, config=MEASURE_CFG)

N_REQUESTS = 50          # per model - 50 here means 150 billable calls total
EFFORT = "low"           # pin it: comparing defaults is not comparing models.
                         # Set to None to omit output_config entirely.
MAX_TOKENS = 512

MODELS = {
    "Haiku 5.5": CRIS_MODEL_ID,
    "Sonnet 5.5": SONNET_MODEL_ID,
    "Opus 5.5": OPUS_MODEL_ID,
}

PROMPT = (
    "Classify this support ticket as BILLING, TECHNICAL, or ACCOUNT. "
    "Reply with one word only.\n\n"
    "Ticket: I was charged twice for my subscription this month."
)

def invoke_once(model_id):
    """One InvokeModel call. Returns (seconds, output_tokens, thinking_tokens).

    InvokeModel rather than Converse: Converse does not report thinking tokens, and
    without them a latency number cannot be interpreted.
    """
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": MAX_TOKENS,
        "messages": [{"role": "user", "content": PROMPT}],
    }
    if EFFORT is not None:
        body["output_config"] = {"effort": EFFORT}
    t0 = time.perf_counter()
    r = rt_measure.invoke_model(modelId=model_id, body=json.dumps(body))
    elapsed = time.perf_counter() - t0
    usage = json.loads(r["body"].read()).get("usage", {})
    thinking = usage.get("output_tokens_details", {}).get("thinking_tokens", 0)
    return elapsed, usage.get("output_tokens", 0), thinking

# Warm up and discard - the first call per model carries TLS and client setup.
for label, model_id in MODELS.items():
    try:
        invoke_once(model_id)
    except Exception as e:
        print(f"warm-up failed for {label}: {type(e).__name__}: {str(e)[:110]}")

samples, failures = defaultdict(list), defaultdict(int)
for i in range(N_REQUESTS):
    for label, model_id in MODELS.items():
        try:
            samples[label].append(invoke_once(model_id))
        except Exception as e:
            failures[label] += 1
            if failures[label] == 1:        # report the first failure, then just count
                print(f"{label}: {type(e).__name__}: {str(e)[:110]}")
    if (i + 1) % 10 == 0:
        print(f"  ...{i + 1}/{N_REQUESTS} rounds")

def pct(values, q):
    """Nearest-rank percentile. At n=50, p95 is the 3rd-slowest sample."""
    s = sorted(values)
    return s[max(0, math.ceil(q / 100 * len(s)) - 1)]

print(f"\neffort={EFFORT}  max_tokens={MAX_TOKENS}  n={N_REQUESTS} per model\n")
header = (f"{'Model':<12}{'mean':>8}{'p50':>8}{'p95':>8}{'min':>8}{'max':>8}"
          f"{'out tok':>9}{'think':>7}{'tok/s':>8}{'fail':>6}")
print(header)
print("-" * len(header))
for label in MODELS:
    rows = samples[label]
    if not rows:
        print(f"{label:<12}  no successful calls ({failures[label]} failed)")
        continue
    lat = [r[0] for r in rows]
    out = statistics.mean(r[1] for r in rows)
    mean_lat = statistics.mean(lat)
    print(f"{label:<12}"
          f"{mean_lat:>7.2f}s{pct(lat, 50):>7.2f}s{pct(lat, 95):>7.2f}s"
          f"{min(lat):>7.2f}s{max(lat):>7.2f}s"
          f"{out:>9.0f}{statistics.mean(r[2] for r in rows):>7.0f}"
          f"{out / mean_lat:>8.1f}{failures[label]:>6}")

print("\ntok/s is mean output tokens over mean latency - a throughput proxy, not a guarantee.")
print("Measured on a no-retry client: a silent SDK retry would add seconds to a sample and")
print("look like model latency. Throttles show up in the 'fail' column instead.")

### Time to first token

For interactive applications, total latency is not what users feel — **time to first token**
(TTFT) is. A model that starts responding in 300ms and finishes in 4s feels faster than one that
stays silent for 2s and finishes in 3s. This is where Haiku should separate hardest.

Fewer requests here because streaming calls are slower to tear down.

In [ ]:
N_TTFT = 10              # per model - 30 calls total

def ttft_once(model_id):
    """Returns (seconds_to_first_text_token, total_seconds)."""
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": 256,
        "messages": [{"role": "user", "content":
                      "In one sentence, what is Amazon Bedrock?"}],
    }
    if EFFORT is not None:
        body["output_config"] = {"effort": EFFORT}
    t0 = time.perf_counter()
    first = None
    r = rt_measure.invoke_model_with_response_stream(
        modelId=model_id, body=json.dumps(body))
    for event in r["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if (chunk["type"] == "content_block_delta"
                and chunk["delta"]["type"] == "text_delta" and first is None):
            first = time.perf_counter() - t0
    return first, time.perf_counter() - t0

ttft = defaultdict(list)
for i in range(N_TTFT):
    for label, model_id in MODELS.items():
        try:
            first, total = ttft_once(model_id)
            if first is not None:
                ttft[label].append((first, total))
        except Exception as e:
            print(f"{label}: {type(e).__name__}: {str(e)[:110]}")

header = f"{'Model':<12}{'TTFT p50':>11}{'TTFT p95':>11}{'total p50':>12}{'n':>5}"
print(f"\n{header}")
print("-" * len(header))
for label in MODELS:
    rows = ttft[label]
    if not rows:
        print(f"{label:<12}  no samples")
        continue
    firsts = [r[0] for r in rows]
    totals = [r[1] for r in rows]
    print(f"{label:<12}{pct(firsts, 50):>10.2f}s{pct(firsts, 95):>10.2f}s"
          f"{pct(totals, 50):>11.2f}s{len(rows):>5}")

print("\nTTFT excludes any thinking the model does before emitting text, so a model with")
print("adaptive thinking on will show a higher TTFT than its raw speed suggests.")

---

## 6. Token efficiency and cost per task

Price per token is the headline, but the unit that matters is **cost per completed task**. A
cheaper model that needs three attempts is not cheaper. Conversely, a model that thinks less per
turn at the same effort finishes the same work for fewer tokens.

The cell below runs one representative task on all three models at the same effort and reports
input, thinking, and output tokens side by side, so you can measure the delta on *your*
workload. Run it on something representative of your production traffic; a single toy prompt is
not a benchmark.


In [ ]:
def usage_for(model_id, prompt, effort="low", max_tokens=8000):
    body = {
        "anthropic_version": ANTHROPIC_VERSION,
        "max_tokens": max_tokens,
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": effort},
        "messages": [{"role": "user", "content": prompt}],
    }
    r = rt.invoke_model(modelId=model_id, body=json.dumps(body))
    res = json.loads(r["body"].read())
    usage = res.get("usage", {})
    text = next((b["text"] for b in res["content"] if b["type"] == "text"), "")
    return {
        "input": usage.get("input_tokens", 0),
        "thinking": usage.get("output_tokens_details", {}).get("thinking_tokens", 0),
        "output": usage.get("output_tokens", 0),
        "chars": len(text),
    }

task = (
    "A service moves 40 TB/month of S3 egress at $0.09/GB and 12 TB/month of cross-AZ "
    "transfer at $0.01/GB. Compute the monthly total, then propose the single change with "
    "the largest saving and quantify it. Show the arithmetic. Under 150 words."
)

print(f"{'model':<34}{'input':>7}{'thinking':>10}{'output':>8}{'chars':>7}")
print("-" * 66)
for label, model_id in MODELS.items():
    try:
        u = usage_for(model_id, task)
        print(f"{label + ' (' + model_id.split('.')[-1] + ')':<34}"
              f"{u['input']:>7}{u['thinking']:>10}{u['output']:>8}{u['chars']:>7}")
    except Exception as e:
        print(f"{label:<34} error: {type(e).__name__}: {str(e)[:40]}")

print("\nCheck the answers, not just the token counts. The cheapest model that gets the")
print("arithmetic right is the right choice for this task; token counts alone cannot tell")
print("you that. Single-run numbers are noisy - loop this if you are making a real decision.")

### 6.1 Task budgets (beta)

Task budgets set a token spending limit for an entire agentic loop, so autonomous runs stay
cost-bounded. As the budget is approached, Claude wraps up gracefully rather than stopping
mid-task. For high-volume Haiku subagents this is the main guardrail against a runaway loop.

- Set the limit via `output_config.task_budget`.
- On Sonnet 5.5 the **minimum budget is 20,000 tokens**; a lower value returns a 400 error.
- Requires a beta header: `anthropic_beta: ["task-budgets-2026-03-13"]`.

In [ ]:
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID,
        max_tokens=4096,
        thinking={"type": "adaptive"},
        extra_body={
            "anthropic_beta": ["task-budgets-2026-03-13"],
            "output_config": {"effort": "low", "task_budget": {"type": "tokens", "total": 20000}},
        },
        messages=[{"role": "user", "content": (
            "Draft a triage runbook for a web service returning intermittent 502s. "
            "Cover first checks, escalation, and rollback criteria."
        )}],
    )
    print(f"Stop reason: {message.stop_reason}")
    print(f"Output tokens used: {message.usage.output_tokens} (budget: 25,000)\n")
    for block in message.content:
        if block.type == "text":
            print(block.text[:800] + "..." if len(block.text) > 800 else block.text)
except Exception as e:
    print(f"Error: {e}")
    print("\nTask budgets require the beta header and may not be enabled for this model")
    print("or Region. See the note above - the beta ID is unverified for Haiku 5.5.")

---

## 7. Prompt caching

Prompt caching reduces latency and cost when you repeatedly send the same large context (system
prompts, documents, few-shot examples) alongside different user queries. It matters
disproportionately for Haiku subagent patterns: if you fan out 50 parallel calls that share one
large system prompt, caching turns 50 full prompt reads into 1 write and 49 cheap reads.

- Mark content cacheable with `"cache_control": {"type": "ephemeral"}`.
- The first request writes the cache; subsequent requests read from it.
- Cache reads are substantially cheaper than reprocessing the same tokens.

> **Note:** Bedrock documents its own minimum cacheable prompt length per model. Do not assume
> the Claude API figure applies — check the Bedrock documentation. If `cache_write` comes back as
> 0 below, your system prompt is likely under the minimum.

In [ ]:
large_system = "You are an expert site reliability engineer.\n" + "\n".join(
    f"Rule {i}: Diagnose from evidence, state the single next check, and never guess "
    f"at a root cause without naming the signal that would confirm it."
    for i in range(1, 400)
)
print(f"System prompt size: ~{len(large_system)} chars\n")

def cached_call(user_msg):
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID,
        max_tokens=64,
        system=[{"type": "text", "text": large_system,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_msg}],
    )
    return message.usage

try:
    u1 = cached_call("A pod is CrashLoopBackOff. What is the first check?")
    write1 = getattr(u1, "cache_creation_input_tokens", 0) or 0
    read1 = getattr(u1, "cache_read_input_tokens", 0) or 0
    print(f"Request 1: cache_write={write1}, cache_read={read1}")

    u2 = cached_call("Disk usage on a node hit 95%. What is the first check?")
    read2 = getattr(u2, "cache_read_input_tokens", 0) or 0
    print(f"Request 2: cache_read={read2}")

    if read2 > 0:
        print(f"\nCache hit: {read2} tokens served from cache on the second request.")
    else:
        print("\nNo cache read. The prompt may be under the minimum cacheable length,")
        print("or the cache entry expired between calls.")
except Exception as e:
    print(f"Error: {e}")

---

## 8. Structured outputs (JSON)

Structured outputs are not supported on Claude Haiku 5.5.
`output_config.format` with a `json_schema` returns
`400 output_config.format: Extra inputs are not permitted`.


In [ ]:
schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["BILLING", "TECHNICAL", "ACCOUNT"]},
        "urgency": {"type": "integer"},
    },
    "required": ["category", "urgency"],
}
ticket = "I was charged twice this month and cannot reach anyone."

# Probe 1: native structured outputs.
print("Probe 1 - output_config.format with json_schema:")
try:
    rt_messages.messages.create(
        model=CRIS_MODEL_ID, max_tokens=256,
        extra_body={"output_config": {"format": {"type": "json_schema", "schema": schema}}},
        messages=[{"role": "user", "content": ticket}],
    )
    print("  accepted - native structured outputs work on this model\n")
except Exception as e:
    print(f"  rejected - {type(e).__name__}: {str(e)[:120]}\n")

# Probe 2: forced tool use, the pre-output_config workaround.
print("Probe 2 - forced tool use (tool_choice):")
classify_tool = [{
    "name": "classify",
    "description": "Record the ticket classification.",
    "input_schema": schema,
}]
for choice in [{"type": "any"}, {"type": "tool", "name": "classify"}]:
    try:
        rt_messages.messages.create(
            model=CRIS_MODEL_ID, max_tokens=256, tools=classify_tool, tool_choice=choice,
            messages=[{"role": "user", "content": ticket}],
        )
        print(f"  {json.dumps(choice)}: accepted")
    except Exception as e:
        print(f"  {json.dumps(choice)}: rejected - {str(e)[:90]}")

---

## 9. Tool use

Tool use lets Claude call functions you define. Haiku 5.5 handles agentic coding and multi-step
tool use, which is what makes it usable as a subagent rather than a single-shot model.

The Messages API and InvokeModel use Anthropic's native tool format; Converse uses Bedrock's
`toolConfig` format.

In [ ]:
tools = [{
    "name": "calculator",
    "description": "Performs basic arithmetic. Use for any math question.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "The math expression"}},
        "required": ["expression"],
    },
}]

print("Messages API, tool use:\n")
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID, max_tokens=512, tools=tools,
        messages=[{"role": "user", "content": "What is 42 * 17? Use the calculator tool."}],
    )
    print(f"Stop reason: {message.stop_reason}")
    print(f"Block types: {[b.type for b in message.content]}")
    for block in message.content:
        if block.type == "tool_use":
            print(f"Tool: {block.name}, Input: {json.dumps(block.input)}")
except Exception as e:
    print(f"Error: {e}")

---

## 10. Context compaction (beta)

Context compaction supports long-running conversations and agentic tasks by automatically
summarising older context as it approaches the context window limit.

- Enable it with a `compact_20260112` edit under `context_management.edits` (passed through
  `extra_body` on the Messages API).
- Set a `trigger` threshold; older context is summarised when input exceeds it.
- The response returns a `compaction` block. **Pass it back in later requests** to maintain
  continuity — dropping it loses the summarised history.
- Requires a beta header: `anthropic_beta: ["compact-2026-01-12"]`.
- On Sonnet 5.5, supported on the Messages API and InvokeModel; Converse returns a validation
  error.

Compaction will not fire on a short conversation — it is built for 100K+ token contexts. The
cell below shows the wiring and prints whether a compaction block came back; on a two-turn
conversation, `False` is the correct and expected result.

> **Unverified for Haiku 5.5:** the beta ID, the edit type, and the per-API support matrix are
> carried over from Sonnet 5.5. Take the real values from the Haiku 5.5 Bedrock test suite — beta
> IDs have diverged between the Claude API docs and Bedrock before.

In [ ]:
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID,
        max_tokens=1024,
        extra_body={
            "anthropic_beta": ["compact-2026-01-12"],
            "context_management": {"edits": [{
                "type": "compact_20260112",
                "trigger": {"type": "input_tokens", "value": 100000},
            }]},
        },
        messages=[{"role": "user", "content": (
            "Let's start a long-running migration project. We are moving 200 microservices "
            "from EC2 to EKS. What are the first architectural decisions to make?"
        )}],
    )
    has_compaction = any(getattr(b, "type", None) == "compaction" for b in message.content)
    print(f"Compaction block returned: {has_compaction}")
    print(f"Block types: {[b.type for b in message.content]}")
    print("(False is expected here - compaction is designed for 100K+ token contexts.)\n")
    for block in message.content:
        if block.type == "text":
            print(block.text[:600] + "..." if len(block.text) > 600 else block.text)
except Exception as e:
    print(f"Error: {e}")
    print("\nCompaction requires the beta header. See the note above - the beta ID is")
    print("unverified for Haiku 5.5.")

---

## 11. High-resolution vision

Haiku 5.5 supports high-resolution image input. Combined with its cost profile, that makes it a
practical computer use subagent for repetitive browser and desktop tasks, and a reasonable first
pass over screenshots, dense documents, charts, and logs.

The image below is generated in-notebook so this cell runs anywhere without an external file.
In production, feed it real high-resolution images — that is where the capability earns its keep.

In [ ]:
import struct, zlib

def create_sample_chart_png():
    """Create a simple 200x100 PNG with coloured bars (simulates a bar chart)."""
    w, h = 200, 100
    bar_colors = [(66, 133, 244), (234, 67, 53), (251, 188, 4), (52, 168, 83)]
    bar_heights = [80, 55, 70, 90]
    bar_width, gap, start_x = 30, 15, 20
    raw = b""
    for y in range(h):
        raw += b"\x00"
        for x in range(w):
            drawn = False
            for i, (bh, color) in enumerate(zip(bar_heights, bar_colors)):
                bx = start_x + i * (bar_width + gap)
                if bx <= x < bx + bar_width and y >= h - bh:
                    raw += bytes(color)
                    drawn = True
                    break
            if not drawn:
                raw += b"\xff\xff\xff"

    def chunk(ct, d):
        c = ct + d
        return struct.pack(">I", len(d)) + c + struct.pack(">I", zlib.crc32(c) & 0xFFFFFFFF)

    ihdr = struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0)
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", ihdr)
            + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b""))

sample_image = create_sample_chart_png()
image_b64 = base64.b64encode(sample_image).decode("utf-8")
print(f"Generated sample chart: {len(sample_image)} bytes\n")

vision_question = "Describe this chart. What type is it, and how many series does it show?"

print("Messages API, vision:\n")
try:
    message = rt_messages.messages.create(
        model=CRIS_MODEL_ID, max_tokens=1024,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png",
                                         "data": image_b64}},
            {"type": "text", "text": vision_question},
        ]}],
    )
    print(next(b.text for b in message.content if b.type == "text"))
except Exception as e:
    print(f"Error: {e}")

---

## 12. Adopting Haiku 5.5

### Migrating from Haiku 4.5

| Step | Action |
|------|--------|
| 1 | Update the model ID to `global.anthropic.claude-haiku-5-5` (CRIS, for `bedrock-runtime`) or `anthropic.claude-haiku-5-5` (bare, for `bedrock-mantle`) |
| 2 | Non-default values are rejected for `temperature`, `top_p`, `top_k`|
| 3 | Replace any `thinking: {"type": "enabled", "budget_tokens": N}` with `thinking: {"type": "adaptive"}` plus `output_config: {"effort": ...}` |
| 4 | Set effort explicitly rather than relying on the default — Haiku 5.5's default effort level is medium |
| 5 | Audit any code that reads `content[0]` — Haiku 5.5 can return a reasoning block first |
| 6 | Re-check IAM and CRIS: confirm your inference profile prefix (`us.`/`eu.`/`au.`/`jp.`/`global.`) is one your account is entitled to |
| 7 | Re-test on representative workloads. Effort controls are new to the Haiku family, so the tuning you did for 4.5 does not carry over |

### Before and after

**Haiku 4.5 request:**
```json
{
  "model": "global.anthropic.claude-haiku-4-5",
  "thinking": {"type": "enabled", "budget_tokens": 4000},
  "temperature": 0.7,
  "messages": []
}
```

**Haiku 5.5 request:**
```json
{
  "model": "global.anthropic.claude-haiku-5-5",
  "thinking": {"type": "adaptive"},
  "output_config": {"effort": "low"},
  "messages": []
}
```


---


### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/what-is-bedrock.html)
- [Anthropic Claude models on Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-anthropic.html)
- [Anthropic Messages API on Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages.html)
- [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference.html)
- [InvokeModel API](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-api.html)
- [Amazon Bedrock pricing](https://aws.amazon.com/bedrock/pricing/)
- [Monitoring Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring.html)
- [Claude Platform on AWS](https://aws.amazon.com/claude-platform/)
- [Getting started notebooks on GitHub](https://github.com/aws-samples/anthropic-on-aws/tree/main/notebooks)